# Lesson 22 — Decorators: Practice (SUPPORTING)

Work in `lessons/lesson-22-decorators/` (point a terminal at that folder).

Before you start, read `lesson.md` — every construct used in these exercises is
taught there. `appt` is always a dict `{"name": str, "phone": str, "time": str}`.
Write all code in `solution.py` first (ships blank), then run each **Driver:** cell.
Every `Expected` block must match byte-exact.

Provided: `solution.py` (blank — the four decorators live here).
`buggy_clinic.py` (provided, buggy — H1b fixes it in place).
You create for the Challenge: `decorators.py`, `clinic.py`, `remind.py`.
Installs: none — stdlib only (`functools`, `time`, `logging`, `pathlib`, `json`).


## E1 — Log every call without changing the return

## Intent
**Wrap any clinic function with entry/exit logging without changing its return.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
In `solution.py`, define `log_call(func)`. The wrapper logs `call <name>`
before the call and `done <name>` after a successful call, at INFO on the
logger named `clinic`. The wrapper forwards `*args, **kwargs` and returns
the body result verbatim.

**File tree**

- `solution.py`

**Interface contract**

- `def log_call(func):` returns a wrapper with `__name__` and `__doc__`
  of `func`.

**Constraints**

- **Use `functools.wraps`.**
- **Log `done` only on success** (exceptions propagate with no `done` line).
- **Never change the return.**

**Coverage**

- Wrapper + passthrough + `wraps` identity → lesson.md Core Concepts 1–3.

**Expected** (Driver cell):
```text
E1 OK
```


In [ ]:
import importlib, logging
import solution; importlib.reload(solution)

records = []
class H(logging.Handler):
    def emit(self, r): records.append(r.getMessage())
log = logging.getLogger("clinic")
log.setLevel(logging.INFO); log.handlers = []; log.addHandler(H())
log.propagate = False

@solution.log_call
def confirm(appt):
    "Mark one appointment confirmed."
    return appt["phone"]

assert confirm({"name": "Ayu", "phone": "081", "time": "09:00"}) == "081"
assert records == ["call confirm", "done confirm"], records
assert confirm.__name__ == "confirm"
assert confirm.__doc__ == "Mark one appointment confirmed."

@solution.log_call
def boom(appt): raise RuntimeError("x")
try:
    boom({})
except RuntimeError: pass
assert records[-1] == "call boom", records
print("E1 OK")

## E2 — Time a function while keeping its identity

## Intent
**Time a clinic function while keeping its name and docstring identical.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
In `solution.py`, define `timed(func)`. Measure wall time with
`time.perf_counter`. The result is returned unchanged. The elapsed seconds
are stored on `wrapper.last_elapsed` (float, `>= 0.0`), set on every call —
success or failure. `wrapper.last_elapsed` starts at `0.0` before any call.

**File tree**

- `solution.py`

**Interface contract**

- `def timed(func):` returns a wrapper exposing `last_elapsed: float`.

**Constraints**

- **Use `functools.wraps`.**
- **Forward `*args, **kwargs`.**
- **Never change the return.**

**Coverage**

- Wall-time + `wraps` identity → lesson.md Core Concepts 2–3.

**Expected** (Driver cell):
```text
E2 OK
```


In [ ]:
@solution.timed
def send(appt, channel="sms"):
    "Send one reminder."
    return channel + ":" + appt["phone"]

assert send({"phone": "081"}, channel="wa") == "wa:081"
assert send.__name__ == "send"
assert send.__doc__ == "Send one reminder."
assert isinstance(send.last_elapsed, float) and send.last_elapsed >= 0.0
print("E2 OK")

## E3 — Reject bad appointments at the boundary

## Intent
**Reject bad appointment data at the boundary before the wrapped function runs.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
In `solution.py`, define `validate_appointment(func)`. Inspect the
appointment — the first positional argument, or the `appt=` keyword when no
positional is given. Raise `ValueError` with these exact strings (non-string
or blank-after-strip counts as empty): `invalid appointment: not a dict`
(not a dict); `invalid name: empty`; `invalid phone: empty`;
`invalid time: empty`. Check in that order. Valid input calls through and
returns the body result.

**File tree**

- `solution.py`

**Interface contract**

- `def validate_appointment(func):` validates, then calls through.

**Constraints**

- **Use `functools.wraps`.**
- **Exact messages verbatim.**
- **Check fields in the listed order.**

**Coverage**

- Boundary validation → lesson.md Engineering Notes (validate innermost).

**Expected** (Driver cell):
```text
E3 OK
```


In [ ]:
import solution
@solution.validate_appointment
def confirm3(appt):
    return "ok:" + appt["phone"]

good = {"name": "Ayu", "phone": "081", "time": "09:00"}
assert confirm3(good) == "ok:081"
assert confirm3(appt=dict(good)) == "ok:081"
cases = [
    (None, "invalid appointment: not a dict"),
    (["x"], "invalid appointment: not a dict"),
    ({"name": "", "phone": "081", "time": "09:00"}, "invalid name: empty"),
    ({"name": "A", "phone": "  ", "time": "09:00"}, "invalid phone: empty"),
    ({"name": "A", "phone": "081", "time": ""}, "invalid time: empty"),
    ({"name": "A", "phone": 0, "time": "09:00"}, "invalid phone: empty"),
]
for bad, msg in cases:
    try:
        confirm3(bad)
    except ValueError as e:
        assert str(e) == msg, (bad, str(e))
    else:
        raise AssertionError(("no raise", bad))
print("E3 OK")

## M1 — Retry a flaky sender, then re-raise

## Intent
**Retry a flaky sender N times, then re-raise the last error unchanged.**

**Type:** REAL-WORLD — Medium — writes 1 file

**Task**
In `solution.py`, define the factory `retry(attempts, exc=(ConnectionError,))`.
It returns a decorator. The wrapper calls the body up to `attempts` times
total, catching only `exc`. First success returns immediately. Errors outside
`exc` are never retried. When every attempt fails, re-raise the last error
object unchanged.

**File tree**

- `solution.py`

**Interface contract**

- `def retry(attempts, exc=(ConnectionError,)):` returns a decorator.

**Constraints**

- **Factory form only — always `@retry(attempts=...)`.**
- **`attempts < 1` raises `ValueError("attempts must be >= 1")` at factory
  call time.**
- **Use `functools.wraps`.**
- **Catch only the `exc` tuple.**

**Coverage**

- Decorator factory (3 layers) → lesson.md Core Concept 4.

**Expected** (Driver cell):
```text
M1 OK
```


In [ ]:
import solution
calls = {"n": 0}
@solution.retry(attempts=3)
def flaky(appt):
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("drop")
    return "sent:" + appt["phone"]
assert flaky({"phone": "089"}) == "sent:089" and calls["n"] == 3

calls2 = {"n": 0}
@solution.retry(attempts=2)
def dead(appt):
    calls2["n"] += 1
    raise ConnectionError("down")
try:
    dead({})
except ConnectionError as e:
    assert str(e) == "down"
else:
    raise AssertionError("no raise")
assert calls2["n"] == 2

calls3 = {"n": 0}
@solution.retry(attempts=3)
def bad(appt):
    calls3["n"] += 1
    raise ValueError("bad number")
try:
    bad({})
except ValueError:
    pass
else:
    raise AssertionError("no raise")
assert calls3["n"] == 1
try:
    solution.retry(0)
except ValueError as e:
    assert str(e) == "attempts must be >= 1"
else:
    raise AssertionError("no raise")
print("M1 OK")

## M2 — Stack log, timing, and validation in order

## Intent
**Compose log, timing, and validation by stacking — with the order observable in the log.**

**Type:** REAL-WORLD — Medium — writes 1 file

**Task**
In the driver cell only, stack your three decorators in exactly this order
(top to bottom): `@log_call`, `@timed`, `@validate_appointment`. A valid call
returns the body value, emits `call` then `done`, keeps `__name__`, and
records `last_elapsed` on the timed layer (reachable as
`__wrapped__.last_elapsed`). An invalid call raises the `ValueError`, emits
`call` with no `done`, and still records elapsed.

**File tree**

- `solution.py` (no new file)

**Constraints**

- **This exact top-to-bottom order.**
- **No new decorator — reuse E1–E3.**

**Coverage**

- Stacking order → lesson.md Core Concept 5.

**Expected** (Driver cell):
```text
M2 OK
```


In [ ]:
import solution
import logging

records2 = []
class H2(logging.Handler):
    def emit(self, r): records2.append(r.getMessage())
log2 = logging.getLogger("clinic")
log2.setLevel(logging.INFO); log2.handlers = []; log2.addHandler(H2())
log2.propagate = False

@solution.log_call
@solution.timed
@solution.validate_appointment
def confirm_m2(appt):
    return "ok:" + appt["phone"]

assert confirm_m2({"name": "Ayu", "phone": "081", "time": "09:00"}) == "ok:081"
assert records2 == ["call confirm_m2", "done confirm_m2"], records2
assert confirm_m2.__name__ == "confirm_m2"
assert confirm_m2.__wrapped__.last_elapsed >= 0.0
records2.clear()
try:
    confirm_m2({"name": "", "phone": "081", "time": "09:00"})
except ValueError as e:
    assert str(e) == "invalid name: empty"
else:
    raise AssertionError("no raise")
assert records2 == ["call confirm_m2"], records2
print("M2 OK")

## H1 — Debug drill: four decorator bugs

## Intent
**Prove your decorators avoided the four classic failures — then write down each root cause.**

**Type:** DEBUG — Hard — no new file

**Task**
The drill plants exactly 4 bugs (one per pattern below). Your `solution.py`
must defeat all four at once. Then fill the 4 write-in slots in the next cell
with expected-vs-observed-vs-root-cause, in your own words.

Patterns:

1. fixed-params wrapper drops keyword calls
2. missing `wraps` lies about `__name__`/`__doc__`
3. wrapper swallows the return (log looks right, value is `None`)
4. retry factory ignores `attempts` (runs once no matter what)

**File tree**

- `solution.py` (no new file)

**Constraints**

- **No new symbols — E1–M1 only.**
- **Write-ins stay yours — one line each, your own words.**

**Bug report** (fill in as you go — see the next cell).

**Coverage**

- The four latent failures → lesson.md Pitfalls 1–4 + `notes/DEBUGGING.md`.

**Expected** (Driver cell):
```text
H1 OK
```


In [ ]:
import solution
@solution.log_call
def h(appt, channel="sms", urgent=False):
    "H1 probe."
    return (appt["phone"], channel, urgent)
assert h({"phone": "081"}, channel="wa", urgent=True) == ("081", "wa", True)
assert h.__name__ == "h" and h.__doc__ == "H1 probe."
assert h({"phone": "082"}) == ("082", "sms", False)
n = {"c": 0}
@solution.retry(attempts=1)
def once(appt):
    n["c"] += 1
    raise ConnectionError("d")
try:
    once({})
except ConnectionError:
    pass
else:
    raise AssertionError("no raise")
assert n["c"] == 1
n2 = {"c": 0}
@solution.retry(attempts=3)
def thrice(appt):
    n2["c"] += 1
    if n2["c"] < 3:
        raise ConnectionError("d")
    return "up"
assert thrice({}) == "up" and n2["c"] == 3
print("H1 OK")

### H1 bug report (fill in after green)

1. kwargs-drop — Expected / Observed / Root cause:

2. missing-wraps — Expected / Observed / Root cause:

3. swallowed-return — Expected / Observed / Root cause:

4. attempts-ignored — Expected / Observed / Root cause:


## H1b — Hunt: four planted bugs in buggy_clinic

## Intent
**Diagnose four real decorator failures in a provided module — fix each where it lives, then write the root cause.**

**Type:** DEBUG — Hard — fixes 1 file

**Task**
`buggy_clinic.py` is provided, buggy: four planted failures, one per H1
pattern (kwargs-drop, missing `wraps`, swallowed return, attempts-ignored).
Nothing else in the file is wrong — exemplary everywhere except the four
plants. Run the DEBUGGING protocol (`notes/DEBUGGING.md`): reproduce via the
driver, read the traceback, locate, hypothesize, fix, explain. Fix in place;
do not restructure the module.

**File tree**

- `buggy_clinic.py` (provided, buggy — YOU fix it)
- `solution.py` (untouched — the hunt stays in `buggy_clinic.py`)

**Interface contract** (fixed states, exact)

- `greet(appt, channel="sms")` forwards kwargs → `"wa:081"`.
- `confirm.__name__ == "confirm"`, docstring preserved.
- `send(appt, channel=...)` returns the delivery string, never `None`.
- `flaky_send` with `@retry(attempts=3)` survives two drops → `"sent:089"`.

**Constraints**

- **Four fixes only — no refactoring, no new symbols.**
- **Each fix stays in its own layer** (wrapper shape, `wraps`, return, loop).
- **Fill the H1 write-in cell from this hunt** (overwrite memory answers with
  hunt evidence where they differ).

**Coverage**

- The four latent failures, for real → lesson.md Pitfalls 1–4.

**Expected** (Driver cell):
```text
H1B OK
```


In [ ]:
import importlib
import buggy_clinic as bc
importlib.reload(bc)

assert bc.greet({"phone": "081"}, channel="wa") == "wa:081"
assert bc.greet({"phone": "082"}) == "sms:082"
assert bc.confirm.__name__ == "confirm"
assert bc.confirm.__doc__ == "Confirm one appointment."
assert bc.send({"phone": "081"}) == "sms:081"
assert bc.send({"phone": "083"}, channel="wa") == "wa:083"
assert bc.flaky_send({"phone": "089"}) == "sent:089"
print("H1B OK")

expected ; observed ; root cause
1. BUG 1: greet should safely receive a param with different length, that is why we have to put asterisk on the '*args' ; wrapper param 'appt' ; didn't use the *args and **kwargs approach, which is primary.
2. BUG 2: assertion passed on __name__ ; missed the functools code ; wrapper didn't wrapped with functools.wraps
3. BUG 3: _deliver result preserved in send ; no return in send ; the _deliver isnt preserved in a var to return it back after logging.
4. BUG 4: retrying as many as attempts set ; no loop for trying ; exception handling didnt wrapped in for loop, and direct return on exception

## Challenge — Morning reminder run for the clinic

## Intent
**Ship a morning reminder run for the clinic: every appointment confirmed, sent, and reported — invalid ones skipped loudly.**

**Type:** REAL-WORLD — Micro Build — writes 3 files

**Task**
The clinic receptionist runs one command each morning:
`python remind.py <inbox.json> <report.txt>`. It reads the day-book JSON,
confirms each appointment, sends each SMS (the gateway drops the first try
for numbers ending in `9`), skips invalid entries with a loud `skip` line,
and writes a report the receptionist can read back — the same text also goes
to stdout. Without decorators this was four copy-pasted scaffolds; with them
it is four `@` lines.

Create `decorators.py`, `clinic.py`, `remind.py` in this folder. Copy your four
tested decorators into `decorators.py` unchanged.

**Flow** (the whole wiring — every arrow is required)

```text
main reads <inbox.json> (always a JSON list of appts)
  → per entry i, in file order:
      confirmed = confirm(appt)      # validates + logs + times
      send_sms(confirmed)            # retries the gateway; ok-line only after it returns
      lines.append("ok <phone>: <name> <time>")
      on ValueError as e: lines.append("skip <i>: <e>")   # no send attempted
  → text = build_report(lines)
  → write text to <report.txt> AND print the same text to stdout
```

The inbox is always a valid JSON list — no corrupt-file handling required.
"Missing inbox" means the path does not exist or is not a file (a directory
counts as missing).

**File tree**

- `decorators.py` (`log_call`, `timed`, `validate_appointment`, `retry`)
- `clinic.py` (`confirm`, `send_sms`, `build_report`)
- `remind.py` (`main`)

**Interface contract**

- `clinic.confirm(appt)` returns the appt dict and carries stacked
  `@log_call` + `@timed` + `@validate_appointment`.
- `clinic.send_sms(appt)` returns `"sent:" + phone` and carries
  `@retry(attempts=3)` outside `@log_call`.
- `clinic.build_report(lines)` returns the header plus lines plus trailing
  newline (header alone when empty). It performs no I/O.
- `remind.main(argv)` — house shape (L20/L21/WP3): receives full `sys.argv`
  (guard is `main(sys.argv)`), paths at `argv[1]` / `argv[2]`, `-> None`,
  errors leave via `sys.exit(1)` after one stderr line.

**Constraints**

- **Same entry shape as L20/L21/WP3: full `sys.argv`, `argv[1]`/`argv[2]`,
  usage to stderr + exit 1 on wrong argc, `-> None`.**
- **`main(argv)` + `__main__` guard, no module-level executable code.**
- **Per entry: `confirm` → `send_sms` → ok-line; `ValueError` → skip-line
  with no send.** (The `089` gateway drop proves the retry ran.)
- **Path-first I/O** (`Path(argv[1])` / `Path(argv[2])`, `read_text` /
  `write_text`, `encoding="utf-8"`).
- **Missing inbox (nonexistent or not a file) prints `bad inbox: not found`
  to stderr and exits 1.**
- **`send_sms` fails the first call only for phones ending in `9` with
  `ConnectionError("gateway drop")`, then succeeds** (module-level seen-set).
- **Stdout and the report file are byte-identical.**

**Coverage**

- All four decorators composed → lesson.md Core Concepts 1–5 +
  entry shape + Path-first I/O (L20/L21 retrieval).

**Expected** (Driver cell, real subprocess runs, fixture of 3):
```text
CH OK
```


In [ ]:
import json, subprocess, sys, tempfile, pathlib, importlib
import clinic
importlib.reload(clinic)
clinic.send_sms.__wrapped__.__wrapped__  # retry outside, log inside

HOME = pathlib.Path.cwd()
inbox_data = [
    {"name": "Ayu", "phone": "081", "time": "09:00"},
    {"name": "Citra", "phone": "", "time": "09:15"},
    {"name": "Budi", "phone": "089", "time": "09:30"},
]
expected = ("confirmed 2, skipped 1\n" "ok 081: Ayu 09:00\n"
    "skip 1: invalid phone: empty\n" "ok 089: Budi 09:30\n")
with tempfile.TemporaryDirectory() as td:
    td = pathlib.Path(td)
    ib, rp = td / "inbox.json", td / "report.txt"
    ib.write_text(json.dumps(inbox_data), encoding="utf-8")
    run = subprocess.run(
        [sys.executable, "remind.py", str(ib), str(rp)],
        capture_output=True, text=True, cwd=HOME)
    assert run.returncode == 0, run.stderr
    assert run.stdout == expected, repr(run.stdout)
    assert rp.read_text(encoding="utf-8") == expected
    usage = subprocess.run(
        [sys.executable, "remind.py", str(ib)],
        capture_output=True, text=True, cwd=HOME)
    assert usage.returncode == 1
    assert usage.stderr == "usage: remind.py <inbox.json> <report.txt>\n"
    missing = subprocess.run(
        [sys.executable, "remind.py", str(td / "nope.json"), str(rp)],
        capture_output=True, text=True, cwd=HOME)
    assert missing.returncode == 1
    assert missing.stderr == "bad inbox: not found\n"
print("CH OK")